In [0]:
df=spark.read.table('retaildataplatform.bronze.cosmosdb_customers')
df.display()

In [0]:
df.printSchema()

In [0]:
 Array of Array

[
    ["AVwjhoD2v8e3D1O-nnNv", "35"], 
    ["AVpgQP5vLJeJML43LQbd", "70"], 
    ["AVwjhoD2v8e3D1O-nnNv", "43"], 
    ["AVwjhoD2v8e3D1O-nnNv", "48"], 
    ["AVqVGaCCU2_QcyX9Ozcf", "23"]
]

-> Array of Struct

[
    {"curr": "USD", "id": "AVwjhoD2v8e3D1O-nnNv", "name": "Ankyo - TX 7.2-Ch. Network-Ready A/V Home Theater Receiver - Black", "price": "2036", "promotion_info": null, "qty": "3", "unit": "pcs"}, 

    {"curr": "USD", "id": "AVqVGaCCU2_QcyX9Ozcf", "name": "15.4 NakBook Pro with Touch Bar (Late 2016, Space Gray)", "price": "376", "promotion_info": null, "qty": "8", "unit": "pcs"}

]



-> Array of json

"promo_info": [
    {"promo_disc": 0.03, "promo_id": "0", "promo_item": "AVpfMVD-ilAPnD_xW6bu", "promo_qty": "2"}
    ]

3 Funtions (Most Important)

from_json -> data type (string) -> structured column
explode -> multiple rows
explode_outer -> same as explode and also handle nulls in data

In [0]:
{"_id": {"$oid": "6a999f4a9e0b277853440d20"},
 "clicked_items": [["AVpgIu4Q1cnluZ0-xBK-", "11"], ["AVpe_semLJeJML430DAU", "14"], ["AVpfCQslilAPnD_xThwe", "17"]], 
 "customer_id": 33899876,
 "customer_name": "SIEGEL,  ADAM", 
 "number_of_line_items": 1, 
 "order_datetime": 1565710733.0, 
 "order_number": 317568501, 
 "ordered_products": [
     {"curr": "USD", 
      "id": "AVpfCQslilAPnD_xThwe", 
      "name": "M80UWH Manual Series Projection Screen (39.6 x 69.6)", "price": "84", 
      "promotion_info": null, 
      "qty": "2", 
      "unit": "pcs"}
     ], 
 "promo_info": []}

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
    IntegerType,
    ArrayType
)

json_schema =  StructType([
    StructField("_id", StructType([
        StructField("$oid", StringType(), True)
    ]), True),

    StructField(
        "clicked_items",
        ArrayType(
            ArrayType(StringType())
        ),
        True
    ),

    StructField("customer_id", LongType(), True),
    StructField("customer_name", StringType(), True),
    StructField("number_of_line_items", IntegerType(), True),
    StructField("order_datetime", DoubleType(), True),
    StructField("order_number", LongType(), True),

    StructField(
        "ordered_products",
        ArrayType(
            StructType([
                StructField("curr", StringType(), True),
                StructField("id", StringType(), True),
                StructField("name", StringType(), True),
                StructField("price", StringType(), True),
                StructField("promotion_info", StringType(), True),
                StructField("qty", StringType(), True),
                StructField("unit", StringType(), True)
            ])
        ),
        True
    ),

    StructField(
        "promo_info",
        ArrayType(StringType()),
        True
    )
])

In [0]:
import pyspark.sql.functions as F
df_parsed=df.withColumn('data',F.from_json(df['json_data'],json_schema))
df_parsed.display()

In [0]:
df_parsed.printSchema()

In [0]:
df_customers=df_parsed.select(
    F.col("data.customer_id").alias('customer_id'),
    F.col('data.customer_name').alias('customer_name'),
    F.col('data.number_of_line_items').alias('number_of_line_items'),
    F.col('data.order_datetime').alias('order_datetime'),
    F.col('data.order_number').alias('order_number'),
    F.col('data.ordered_products').alias('ordered_products'),
    F.explode(F.col('data.ordered_products')).alias('products'),
    F.explode(F.col('data.promo_info')).alias('promo_info')
)
df_customers.display()

In [0]:
df_customers.printSchema()

In [0]:
df_customers=df_customers.select(
    F.col('customer_id'),
    F.col('customer_name'),
    F.col('number_of_line_items'),
    F.col('order_datetime'),
    F.col('products.curr').alias('curr'),
    F.col('products.id').alias('id'),
    F.col('products.name').alias('name'),
    F.col('products.price').alias('price'),
    F.col('products.promotion_info').alias('promotion_info'),
    F.col('products.qty').alias('qty'),
    F.col('products.unit').alias('unit')
)

df_customers.display()

In [0]:
df_clicked_items=df_parsed.select(
    F.col('data.customer_id').alias('customer_id'),
    F.explode(F.col('data.clicked_items')).alias('clicked_items')
)
df_clicked_items.display()

In [0]:
df_clicked_items=df_clicked_items.select(
    F.col('customer_id').alias('customer_id'),
    F.col('clicked_items')[0].alias('product_id'),
    F.col('clicked_items')[1].alias('score')
)
df_clicked_items.display()

### 1. Sales Orders Table

In [0]:
sales_orders=df_customers.select(
    F.col('order_number'),
    F.col('customer_id'),
    F.col('customer_name'),
    F.col('number_of_line_items'),
    F.col('order_datetime').alias('order_timestamp')
)
sales_orders.display()

### 2. Order Products Table

In [0]:
order_products=df_customers.select(
    F.col('customer_id'),
    F.col('customer_name'),
    F.col('order_number'),
    F.col('products.id').alias('product_id'),
    F.col('products.name').alias('product_name'),
    F.col('products.price').alias('price'),
    F.col('products.curr').alias('currency'),
    F.col('products.qty').alias('quantity'),
    F.col('products.unit').alias('unit')
)
order_products.display()

### 3. Promotions Table

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

promo_schema = StructType([
    StructField("promo_disc", DoubleType(), True),
    StructField("promo_id", StringType(), True),
    StructField("promo_item", StringType(), True),
    StructField("promo_qty", StringType(), True)
])

df_customers = df_parsed.select(
    F.col("data.customer_id").alias("customer_id"),
    F.col("data.order_number").alias("order_number"),
    F.explode_outer(F.col("data.promo_info")).alias("promo_info_str")
).withColumn(
    "promo_info", F.from_json(F.col("promo_info_str"), promo_schema)
).select(
    F.col("customer_id"),
    F.col("order_number"),
    F.col("promo_info.promo_id").alias("promo_id"),
    F.col("promo_info.promo_item").alias("promo_product_id"),
    F.col("promo_info.promo_disc").alias("discount"),
    F.col("promo_info.promo_qty").alias("promo_quantity")
)

df_customers.display()

### 4. Clicked Itmes Table

In [0]:
clicked_items = df_parsed.select(
    F.col('data.customer_id').alias('customer_id'),
    F.col('data.customer_name').alias('customer_name'),
    F.col('data.order_number').alias('order_number')
).join(
    df_clicked_items,
    'customer_id',
    'left'
).select(
    F.col('customer_id'),
    F.col('customer_name'),
    F.col('order_number'),
    F.col('product_id'),
    F.col('score')
)

clicked_items.display()

#### Sales Table

In [0]:
df=spark.read.table('retaildataplatform.bronze.s3sales_customers')
df.display()

In [0]:
df.printSchema()

In [0]:
product_schema = StructType([
    StructField("curr", StringType()),
    StructField("id", StringType()),
    StructField("name", StringType()),
    StructField("price", DoubleType()),
    StructField("qty", IntegerType()),
    StructField("unit", StringType()),
    
])
sales_df=df.withColumn('product',F.from_json('product',product_schema)).select(
     "customer_id",
    "customer_name",
    "product_name", 
    "order_date",
    "product_category",
    "product.curr",
    "product.id",
    "product.name",
    "product.price",
    "product.qty",
    "product.unit",
    "total_price",
    "last_updated_ts"
)

In [0]:
sales_df.display()